# RNAscope Reanalysis

Batch reanalyse saved RNAscope ROI JSON files from the NWB session folders, save new reanalysis JSON files, and compare the reconstructed counts with the experimenter counts stored in each NWB file.

In [ ]:
from pathlib import Path
import json
import re
import sys

import pandas as pd
from IPython.display import display, Markdown
from pynwb import NWBHDF5IO

repo = Path("/Users/euo9382/Documents/Repositories/analysis_Belal2026")
functions_dir = repo / "Python functions"

if str(functions_dir) not in sys.path:
    sys.path.insert(0, str(functions_dir))

from master_RNAscope import (
    RNAscopeAnalysisFinish,
    reconstruct_state_from_saved_analysis,
    save_rnascope_field_analysis,
    parse_field_metadata,
    counts_from_roi_jsons,
)

nwb_root = repo / "NWBdata" / "001832"

sessions = ["sub-L1-ST8_ses-20240905T115902", 
            "sub-L2-ST8_ses-20240909T100245", 
            "sub-L3-ST6_ses-20240909T112846", 
            "sub-L4-ST8_ses-20240916T101347"]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

display(Markdown(f"NWB root: `{nwb_root}`"))

In [ ]:
# Reanalysis Settings

run_reanalysis = True
save_reanalysis = True
source_analysis_folder = "analysis"
reanalysis_folder = "reanalysis"
display_mode = "rendered_from_raw"

detection_method = "DoG"
dog_mode = "tolerance"

if detection_method == "DoG":
    new_params = {
        "detection_method": detection_method,
        "dog_mode": dog_mode,
        "sigma_small": 1.0,
        "sigma_large": 2.8,
        "threshold_percentile": 99,
        "peak_footprint": 4,
        "maxima_tolerance": 170,
        "show_detected": False,
        "show_verify": False,
    }
else:
    new_params = {
        "detection_method": detection_method,
        "maxima_tolerance": 160,
        "show_detected": False,
        "show_verify": False,
    }
    
new_params

In [ ]:
# Check Inputs
input_rows = []

session_paths = {
    "L1.ST8": {
        "session_root": nwb_root / "sub-L1-ST8",
        "nwb_path": nwb_root / "sub-L1-ST8" / "sub-L1-ST8_ses-20240905T115902.nwb",
    },
    "L2.ST8": {
        "session_root": nwb_root / "sub-L2-ST8",
        "nwb_path": nwb_root / "sub-L2-ST8" / "sub-L2-ST8_ses-20240909T100245.nwb",
    },
    "L3.ST6": {
        "session_root": nwb_root / "sub-L3-ST6",
        "nwb_path": nwb_root / "sub-L3-ST6" / "sub-L3-ST6_ses-20240909T112846.nwb",
    },
    "L4.ST8": {
        "session_root": nwb_root / "sub-L4-ST8",
        "nwb_path": nwb_root / "sub-L4-ST8" / "sub-L4-ST8_ses-20240916T101347.nwb",
    },
}

for session, paths in session_paths.items():
    session_root = paths["session_root"]
    nwb_path = paths["nwb_path"]
    analysis_dir = session_root / source_analysis_folder
    json_paths = sorted(analysis_dir.glob("*.roi_analysis.json"))

    input_rows.append(
        {
            "session": session,
            "nwb_exists": nwb_path.exists(),
            "analysis_dir_exists": analysis_dir.exists(),
            "n_json": len(json_paths),
            "nwb_path": str(nwb_path),
            "analysis_dir": str(analysis_dir),
        }
    )

input_summary = pd.DataFrame(input_rows)
display(input_summary)

missing = input_summary[
    (~input_summary["nwb_exists"])
    | (~input_summary["analysis_dir_exists"])
    | (input_summary["n_json"] == 0)
]

if not missing.empty:
    raise FileNotFoundError("Missing NWB files or analysis JSONs. See input_summary above.")

In [ ]:
# Run Reanalysis
run_rows = []

if run_reanalysis:
    for session, paths in session_paths.items():
        session_root = paths["session_root"]
        nwb_path = paths["nwb_path"]
        analysis_dir = session_root / source_analysis_folder
        reanalysis_dir = session_root / reanalysis_folder
        json_paths = sorted(analysis_dir.glob("*.roi_analysis.json"))

        if save_reanalysis:
            reanalysis_dir.mkdir(parents=True, exist_ok=True)

        print(f"{session}: reanalysing {len(json_paths)} fields")

        for json_path in json_paths:
            state, analysis_payload = reconstruct_state_from_saved_analysis(
                nwb_path,
                analysis_path=json_path,
                display_mode=display_mode,
            )

            results, state = RNAscopeAnalysisFinish(state, **new_params)

            out_path = None
            if save_reanalysis:
                out_path = save_rnascope_field_analysis(
                    state,
                    results,
                    analysis_params=new_params,
                    out_dir=reanalysis_dir,
                )

            run_rows.append(
                {
                    "session": session,
                    "field": state["field"],
                    "count_channel": state.get("count_channel"),
                    "n_roi_rows": len(results),
                    "source_json": str(json_path),
                    "reanalysis_json": str(out_path) if out_path is not None else None,
                }
            )

run_summary = pd.DataFrame(run_rows)
display(run_summary)

In [ ]:
# Build Counts DataFrame From Reanalysis JSONs
import re
import master_RNAscope

master_RNAscope.FIELD_RE = re.compile(
    r"^(?P<slice_id>.+?)[._](?P<hemisphere>UL|L)_60x\.?(?P<field_index>\d+)$"
)

reanalysis_json_paths = []

for session, paths in session_paths.items():
    reanalysis_dir = paths["session_root"] / reanalysis_folder
    reanalysis_json_paths.extend(sorted(reanalysis_dir.glob("*.roi_analysis.json")))

reanalysis_counts = counts_from_roi_jsons(reanalysis_json_paths)

display(Markdown(f"Reanalysis JSON files: `{len(reanalysis_json_paths)}`"))
display(reanalysis_counts)

## Load Experimenter Counts From NWB

In [ ]:
experimenter_dfs = []

for session, paths in session_paths.items():
    nwb_path = paths["nwb_path"]

    with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
        nwbfile = io.read()
        df = (
            nwbfile.processing["rnascope_analysis_metadata"]["experimenter_chrnb2_counts"]
            .to_dataframe()
            .reset_index(drop=True)
        )

    df["session"] = df["session"] if "session" in df.columns else session
    df["session_group"] = df["session_group"] if "session_group" in df.columns else session
    experimenter_dfs.append(df)

experimenter_counts = pd.concat(experimenter_dfs, ignore_index=True)
experimenter_counts["field_index"] = experimenter_counts["field_index"].astype(int)
experimenter_counts["replicate"] = experimenter_counts["replicate"].astype(int)
experimenter_counts["count"] = experimenter_counts["count"].astype(int)

display(experimenter_counts)

In [ ]:
# Compare Reanalysis Counts With Experimenter Counts

session_name_map = {
    "sub-L1-ST8": "L1.ST8",
    "sub-L2-ST8": "L2.ST8",
    "sub-L3-ST6": "L3.ST6",
    "sub-L4-ST8": "L4.ST8",
}

reanalysis_counts["session"] = reanalysis_counts["session"].replace(session_name_map)
reanalysis_counts["session_group"] = reanalysis_counts["session_group"].replace(session_name_map)

compare_keys = [
    "condition",
    "cell_type",
    "slice_id",
    "field",
    "hemisphere",
    "field_index",
    "replicate",
    "session",
    "session_group",
]

exact_compare = experimenter_counts[compare_keys + ["count"]].merge(
    reanalysis_counts[compare_keys + ["count", "count_channel"]],
    on=compare_keys,
    how="outer",
    suffixes=("_experimenter", "_reanalysis"),
    indicator=True,
)

exact_compare["count_delta"] = exact_compare["count_reanalysis"] - exact_compare["count_experimenter"]

display(Markdown("### Exact ROI/replicate comparison"))
display(exact_compare)

display(Markdown("### Merge status"))
display(exact_compare["_merge"].value_counts(dropna=False).rename_axis("merge_status").reset_index(name="n"))

matched = exact_compare[exact_compare["_merge"] == "both"].copy()
if not matched.empty:
    pearson_r = matched["count_experimenter"].corr(matched["count_reanalysis"], method="pearson")
    spearman_r = matched["count_experimenter"].corr(matched["count_reanalysis"], method="spearman")
    print("matched ROI rows:", len(matched))
    print("Pearson r:", round(pearson_r, 4))
    print("Spearman r:", round(spearman_r, 4))
    print("mean delta:", round(matched["count_delta"].mean(), 4))
    print("median delta:", round(matched["count_delta"].median(), 4))

In [ ]:
# Field-Level Mean Comparison

field_keys = [
    "condition",
    "cell_type",
    "slice_id",
    "field",
    "hemisphere",
    "field_index",
    "session",
    "session_group",
]

experimenter_field = (
    experimenter_counts
    .groupby(field_keys, as_index=False)["count"]
    .mean()
    .rename(columns={"count": "experimenter_mean"})
)

reanalysis_field = (
    reanalysis_counts
    .groupby(field_keys, as_index=False)["count"]
    .mean()
    .rename(columns={"count": "reanalysis_mean"})
)

field_compare = experimenter_field.merge(
    reanalysis_field,
    on=field_keys,
    how="outer",
    indicator=True,
)
field_compare["mean_delta"] = field_compare["reanalysis_mean"] - field_compare["experimenter_mean"]

display(field_compare)

matched_field = field_compare[field_compare["_merge"] == "both"].copy()
if not matched_field.empty:
    pearson_r = matched_field["experimenter_mean"].corr(matched_field["reanalysis_mean"], method="pearson")
    spearman_r = matched_field["experimenter_mean"].corr(matched_field["reanalysis_mean"], method="spearman")
    print("matched field rows:", len(matched_field))
    print("Pearson r:", round(pearson_r, 4))
    print("Spearman r:", round(spearman_r, 4))

In [ ]:
# Optional CSV Export
save_csv = False

if save_csv:
    counts_csv = nwb_root / "rnascope_reanalysis_counts.csv"
    exact_compare_csv = nwb_root / "rnascope_reanalysis_vs_experimenter_exact.csv"
    field_compare_csv = nwb_root / "rnascope_reanalysis_vs_experimenter_field_means.csv"

    reanalysis_counts.to_csv(counts_csv, index=False)
    exact_compare.to_csv(exact_compare_csv, index=False)
    field_compare.to_csv(field_compare_csv, index=False)

    print(counts_csv)
    print(exact_compare_csv)
    print(field_compare_csv)